# Russian Edit Corrector Pipeline

Top-to-bottom entrypoint for config loading, data preparation, synthetic generation, labels, training scaffold, evaluation, reports, and manual examples.

In [1]:
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / 'src').exists() else cwd.parent
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import load_config

config_path = project_root / 'configs' / 'config.yaml'
config = load_config(config_path)
config['project']['name']

'russian-edit-corrector'

In [2]:
import sys
import torch

env = {
    'python': sys.version.split()[0],
    'cuda_available': torch.cuda.is_available(),
    'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
}
env

{'python': '3.10.12',
 'cuda_available': True,
 'device': 'NVIDIA GeForce RTX 4070'}

In [3]:
import importlib
import src.data.full_dataset_builder as fdb

importlib.reload(fdb)

dataset_build = fdb.build_dataset_from_config(config, force=True)
dataset_build

Loading clean corpus cache: data/raw/clean_corpus_sentences.txt.gz
Loaded 105569 clean sentences from cache


{'status': 'built',
 'path': 'data/processed/correction_dataset.csv.gz',
 'manifest_path': 'reports/dataset_manifest.json',
 'total': 450000,
 'composition': {'clean': 45000, 'synthetic': 403403, 'real': 1597},
 'splits': {'test': 22500, 'val': 22501, 'train': 404999},
 'clean_corpus': {'count': 105569,
  'cache_path': 'data/raw/clean_corpus_sentences.txt.gz',
  'source_counts': {'cache': 105569}}}

In [ ]:
import pandas as pd

dataset_frame = pd.read_csv(config['data']['processed_train_path'])
examples = dataset_frame.head(6).to_dict('records')
dataset_frame.shape, dataset_frame['split'].value_counts().to_dict(), examples[:2]


In [8]:
from src.alignment.aligner import Aligner

aligner = Aligner()
[(row['source'], aligner.align(row['source'], row['target']).is_supported) for row in examples]


[('врядли это сложный результат для документа 254813', True),
 ('кое как работает корпус 33069 но результат важен', True),
 ('кто нибудь проверит раздел 336868 если будет время', True),
 ('Сегодня важный день потому что готов раздел 330084', True),
 ('Я незнаю что делать с словарь 348190', True),
 ('Мы проверяем что то важное в разделе 42775', True)]

In [9]:
from src.training.train import train

train_result = train(config_path)
train_result

Building training features:   0%|                                                                             …

Training setup: 10000 examples, 5000 micro-steps/epoch, 625 optimizer steps/epoch, accumulation=8


Training epoch:   0%|                                                                                         …

Training epoch:   0%|                                                                                         …

Training epoch:   0%|                                                                                         …

Evaluating 2000 examples with TrainedModelCorrector...


Evaluating:   0%|                                                                                             …

Reports written to: reports


{'status': 'trained',
 'output_dir': 'models/adapters/latest',
 'feature_count': 10000,
 'model_training_ran': True,
 'train_loss': 0.04815675782146864,
 'evaluation_count': 2000,
 'evaluation_metrics': {'exact_match': 0.527,
  'dirty_improved_rate': 0.4738598442714127,
  'dirty_worse_rate': 0.5255839822024472,
  'clean_overcorrection_rate': 0.0,
  'edit_precision': 1.0,
  'edit_recall': 0.7857142857142857,
  'edit_f1': 0.88,
  'spelling_precision': 1.0,
  'spelling_recall': 0.75,
  'spelling_f1': 0.8571428571428571,
  'punctuation_precision': 1.0,
  'punctuation_recall': 1.0,
  'punctuation_f1': 1.0},
 'reports_dir': 'reports',
 'report_paths': {'dataset_report.md': 'reports/dataset_report.md',
  'training_report.md': 'reports/training_report.md',
  'evaluation_summary.csv': 'reports/evaluation_summary.csv',
  'error_by_type.csv': 'reports/error_by_type.csv',
  'clean_overcorrection_examples.csv': 'reports/clean_overcorrection_examples.csv',
  'dirty_worse_examples.csv': 'reports/dirt

In [10]:
from src.inference.corrector import Corrector
from src.inference.model_corrector import TrainedModelCorrector
from src.evaluation.metrics import compute_metrics

if 'examples' not in globals():
    import pandas as pd
    dataset_frame = pd.read_csv(config['data']['processed_train_path'])
    examples = dataset_frame.head(6).to_dict('records')

adapter_dir = project_root / config['paths']['adapter_output_dir']
heads_path = project_root / config['paths']['heads_output_dir'] / 'heads.pt'
use_trained_corrector = bool(train_result.get('model_training_ran')) and adapter_dir.exists() and heads_path.exists()
model_corrector = TrainedModelCorrector.from_config(config) if use_trained_corrector else Corrector()
corrector_kind = 'trained_model' if use_trained_corrector else 'rule_fallback'
rows = []
for row in examples:
    rows.append({
        'source': row['source'],
        'target': row['target'],
        'prediction': model_corrector.correct(row['source']).corrected_text,
        'is_clean': bool(row['is_clean']),
    })
corrector_kind, compute_metrics(rows)


('trained_model',
 {'exact_match': 0.5,
  'dirty_improved_rate': 0.5,
  'dirty_worse_rate': 0.5,
  'clean_overcorrection_rate': 0.0,
  'edit_precision': 1.0,
  'edit_recall': 0.8888888888888888,
  'edit_f1': 0.9411764705882353,
  'spelling_precision': 1.0,
  'spelling_recall': 0.8571428571428571,
  'spelling_f1': 0.923076923076923,
  'punctuation_precision': 1.0,
  'punctuation_recall': 1.0,
  'punctuation_f1': 1.0})

In [11]:
if 'model_corrector' not in globals():
    from src.inference.model_corrector import TrainedModelCorrector
    model_corrector = TrainedModelCorrector.from_config(config)
    corrector_kind = 'trained_model'

manual = ['Я незнаю что делать', 'сегодня что то произошло', 'Я люблю этот дом']
corrector_kind, [(text, model_corrector.correct(text).corrected_text) for text in manual]

('trained_model',
 [('Я незнаю что делать', 'Я незнаю, что делать.'),
  ('сегодня что то произошло', 'Сегодня, что-то произошло.'),
  ('Я люблю этот дом', 'Я люблю этот дом.')])